# NYISO Solar Forecasting: Physics-Informed Feature Engineering for Residual Learning

## Imports and Configuration

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_NYISOSolarForecast"

data_root = repo_root / "data"
processed_dir = data_root / "processed"

merged_out = processed_dir / "03_merged_data.csv"

df = pd.read_csv(merged_out, low_memory=False)

## Standardize Columns 

In [3]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df["time_stamp"] = pd.to_datetime(df["time_stamp"], utc=True, errors="coerce")

if "time" in df.columns:
    df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
    same_time_mask = (df["time"] == df["time_stamp"]) | (
        df["time"].isna() & df["time_stamp"].isna()
    )
    if bool(same_time_mask.all()):
        df = df.drop(columns=["time"])

numeric_cols = [
    "actual_mw",
    "forecast_mw",
    "capacity_mw",
    "temperature_2m",
    "surface_pressure",
    "cloud_cover",
    "windspeed_10m",
    "shortwave_radiation",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["zone_name"] = df["zone_name"].astype(str).str.strip().str.upper()

## Time and Error Metric Features

In [4]:
df["time_local"] = df["time_stamp"].dt.tz_convert("America/New_York")


df["year_from_ts"] = df["time_local"].dt.year
df["month_local"] = df["time_local"].dt.month
df["dayofweek_local"] = df["time_local"].dt.dayofweek
df["hour_local"] = df["time_local"].dt.hour

df["is_weekend"] = df["dayofweek_local"].isin([5, 6]).astype(int)
df["is_daylight_proxy"] = (df["shortwave_radiation"] > 0).astype(int)

In [5]:
df["forecast_error_mw"] = df["actual_mw"] - df["forecast_mw"]
df["absolute_error_mw"] = (df["actual_mw"] - df["forecast_mw"]).abs()
df["smape"] = np.where(
    (df["actual_mw"].abs() + df["forecast_mw"].abs()) > 0,
    200 * df["absolute_error_mw"] / (df["actual_mw"].abs() + df["forecast_mw"].abs()),
    np.nan
)

## SYSTEM-Only Modeling Table

In [6]:
df_system = (
    df[df["zone_name"] == "SYSTEM"]
    .copy()
    .sort_values("time_stamp")
    .reset_index(drop=True)
)

print("System Shape:", df_system.shape)
print("System Time Range:", df_system["time_stamp"].min(), "to", df_system["time_stamp"].max())
print("System Columns:", df_system.columns.tolist())

System Shape: (42455, 21)
System Time Range: 2020-11-17 05:00:00+00:00 to 2025-09-21 03:00:00+00:00
System Columns: ['time_stamp', 'zone_name', 'actual_mw', 'forecast_mw', 'capacity_mw', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'year', 'time_local', 'year_from_ts', 'month_local', 'dayofweek_local', 'hour_local', 'is_weekend', 'is_daylight_proxy', 'forecast_error_mw', 'absolute_error_mw', 'smape']


### Encoding Cyclic Time 
solar gen is periodic with 24 hours and 365 days, sin(theta) = sin((2*pi*t)/T)

In [7]:
df_system["dayofyear_local"] = df_system["time_local"].dt.dayofyear
df_system["weekofyear_local"] = df_system["time_local"].dt.isocalendar().week.astype(int)

df_system["hour_sin"] = np.sin(2 * np.pi * df_system["hour_local"] / 24)
df_system["hour_cos"] = np.cos(2 * np.pi * df_system["hour_local"] / 24)

df_system["month_sin"] = np.sin(2 * np.pi * df_system["month_local"] / 12)
df_system["month_cos"] = np.cos(2 * np.pi * df_system["month_local"] / 12)

df_system["dayofyear_sin"] = np.sin(2 * np.pi * df_system["dayofyear_local"] / 365.25)
df_system["dayofyear_cos"] = np.cos(2 * np.pi * df_system["dayofyear_local"] / 365.25)

df_system["dayofweek_sin"] = np.sin(2 * np.pi * df_system["dayofweek_local"] / 7)
df_system["dayofweek_cos"] = np.cos(2 * np.pi * df_system["dayofweek_local"] / 7)

### Solar Regimes

In [8]:
df_system["is_morning_ramp"] = df_system["hour_local"].between(6, 9).astype(int)
df_system["is_midday"] = df_system["hour_local"].between(10, 14).astype(int)
df_system["is_evening_ramp"] = df_system["hour_local"].between(15, 18).astype(int)
df_system["is_night"] = (df_system["is_daylight_proxy"] == 0).astype(int)

### Understanding Capacity 
capacity ratio is forecast/installed cap

In [9]:
df_system["capacity_ffill"] = df_system["capacity_mw"].ffill().bfill()

df_system["forecast_capacity_ratio"] = np.where(
    df_system["capacity_ffill"] > 0,
    df_system["forecast_mw"] / df_system["capacity_ffill"],
    np.nan
)

if "capacity_mw" in df_system.columns:
    df_system["capacity_ffill_safe"] = df_system["capacity_mw"].ffill()
    df_system["capacity_missing_flag"] = df_system["capacity_mw"].isna().astype(int)

    df_system["forecast_capacity_ratio_safe"] = np.where(
        df_system["capacity_ffill_safe"].notna() & (df_system["capacity_ffill_safe"] > 0),
        df_system["forecast_mw"] / df_system["capacity_ffill_safe"],
        np.nan
    )

### Physics-Informed Features Interactions
PV depends on irradiance, temp, and how clouds move
P= f(G,T,C)

In [10]:
df_system["cloud_cover_frac"] = df_system["cloud_cover"] / 100.0

df_system["forecast_mw_log1p"] = np.log1p(df_system["forecast_mw"].clip(lower=0))
df_system["shortwave_radiation_log1p"] = np.log1p(df_system["shortwave_radiation"].clip(lower=0))

In [11]:
interaction_features = pd.DataFrame(index=df_system.index)

interaction_features["forecast_x_hour_sin"] = df_system["forecast_mw"] * df_system["hour_sin"]
interaction_features["forecast_x_hour_cos"] = df_system["forecast_mw"] * df_system["hour_cos"]

interaction_features["shortwave_x_cloud"] = (
    df_system["shortwave_radiation"] * df_system["cloud_cover_frac"]
)

interaction_features["shortwave_x_temp"] = (
    df_system["shortwave_radiation"] * df_system["temperature_2m"]
)

interaction_features["temp_x_wind"] = (
    df_system["temperature_2m"] * df_system["windspeed_10m"]
)

df_system = pd.concat([df_system, interaction_features], axis=1)

### Rolling Windows and Ramp Features

In [12]:
rolling_windows = [3, 6, 24]

rolling_features = {}

for w in rolling_windows:

    rolling_features[f"forecast_roll_mean_{w}"] = (
        df_system["forecast_mw"].shift(1).rolling(w, min_periods=1).mean()
    )

    rolling_features[f"shortwave_roll_mean_{w}"] = (
        df_system["shortwave_radiation"].shift(1).rolling(w, min_periods=1).mean()
    )

rolling_df = pd.DataFrame(rolling_features)

df_system = pd.concat([df_system, rolling_df], axis=1)

In [13]:
df_system["forecast_diff_1"] = df_system["forecast_mw"].diff(1)
df_system["shortwave_diff_1"] = df_system["shortwave_radiation"].diff(1)

df_system["forecast_ramp_abs"] = df_system["forecast_diff_1"].abs()
df_system["shortwave_ramp_abs"] = df_system["shortwave_diff_1"].abs()

## Missingness Summary

In [14]:
feature_missing_summary = (
    pd.DataFrame({
        "column": df_system.columns,
        "dtype": [str(df_system[c].dtype) for c in df_system.columns],
        "missing_count": [df_system[c].isna().sum() for c in df_system.columns],
    })
    .assign(missing_pct=lambda x: 100 * x["missing_count"] / len(df_system))
    .sort_values(["missing_count", "column"], ascending=[False, True])
)

feature_missing_summary.head(40)

,column,dtype,missing_count,missing_pct
36,forecast_capacity_ratio,float64,26504,62.428454
39,forecast_capacity_ratio_safe,float64,26504,62.428454
37,capacity_ffill_safe,float64,23903,56.301967
4,capacity_mw,float64,23903,56.301967
20,smape,float64,18910,44.541279
19,absolute_error_mw,float64,1000,2.355435
18,forecast_error_mw,float64,1000,2.355435
2,actual_mw,float64,712,1.677070
54,forecast_diff_1,float64,301,0.708986
56,forecast_ramp_abs,float64,301,0.708986


## Modeling Dataset
The target is the residual, the error. Also columns that could cause leakage needs to be dropped.

In [20]:
target = "forecast_error_mw"

drop_cols = [
    "actual_mw",
    "absolute_error_mw",
    "smape",
    "time_stamp",
    "time_local",
    "date_local",
    "zone_name",
    "forecast_error_mw",
    "year",
]

drop_cols_existing = [c for c in drop_cols if c in df_system.columns]
feature_cols = [c for c in df_system.columns if c not in drop_cols_existing]

X = df_system[feature_cols].copy()
y = df_system[target].copy()

In [21]:
print("Target:", target)
print("Number of Features:", len(feature_cols))
print("Feature Columns:")
print(feature_cols)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nX dtypes:")
print(X.dtypes.sort_index())

print("\nTop NaN Counts in X:")
print(X.isna().sum().sort_values(ascending=False).head(30))

print("\nTarget NaN Count:")
print(y.isna().sum())

print("\nEvaluation Columns NaN Counts:")
print(df_system[["actual_mw", "forecast_mw", target]].isna().sum())

Target: forecast_error_mw
Number of Features: 50
Feature Columns:
['forecast_mw', 'capacity_mw', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'year_from_ts', 'month_local', 'dayofweek_local', 'hour_local', 'is_weekend', 'is_daylight_proxy', 'dayofyear_local', 'weekofyear_local', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dayofyear_sin', 'dayofyear_cos', 'dayofweek_sin', 'dayofweek_cos', 'is_morning_ramp', 'is_midday', 'is_evening_ramp', 'is_night', 'capacity_ffill', 'forecast_capacity_ratio', 'capacity_ffill_safe', 'capacity_missing_flag', 'forecast_capacity_ratio_safe', 'cloud_cover_frac', 'forecast_mw_log1p', 'shortwave_radiation_log1p', 'forecast_x_hour_sin', 'forecast_x_hour_cos', 'shortwave_x_cloud', 'shortwave_x_temp', 'temp_x_wind', 'forecast_roll_mean_3', 'shortwave_roll_mean_3', 'forecast_roll_mean_6', 'shortwave_roll_mean_6', 'forecast_roll_mean_24', 'shortwave_roll_mean_24', 'forecast_diff_1', 'shortwave_diff_1', 'forec

## Train/Test Split 

In [22]:
split_date = pd.Timestamp("2025-01-01 00:00:00+00:00")

train_time_mask = df_system["time_stamp"] < split_date
test_time_mask = df_system["time_stamp"] >= split_date

train_model_mask = train_time_mask & y.notna()
test_model_mask = (
    test_time_mask
    & y.notna()
    & df_system["actual_mw"].notna()
    & df_system["forecast_mw"].notna()
)

X_train = X.loc[train_model_mask].copy()
X_test = X.loc[test_model_mask].copy()

y_train = y.loc[train_model_mask].copy()
y_test = y.loc[test_model_mask].copy()

baseline_actual_test = df_system.loc[test_model_mask, "actual_mw"].copy()
baseline_forecast_test = df_system.loc[test_model_mask, "forecast_mw"].copy()

In [29]:
print("Split date:", split_date)
print("Train time rows:", int(train_time_mask.sum()))
print("Test time rows:", int(test_time_mask.sum()))

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTop NaN counts in X_train:")
print(X_train.isna().sum().sort_values(ascending=False).head(30))

print("\nTop NaN counts in X_test:")
print(X_test.isna().sum().sort_values(ascending=False).head(30))

print("\ny_train NaN count:", int(y_train.isna().sum()))
print("y_test NaN count:", int(y_test.isna().sum()))

print("\nBaseline Shapes:")
print("baseline_actual_test:", baseline_actual_test.shape)
print("baseline_forecast_test:", baseline_forecast_test.shape)

print("\nBaseline Input NaN Counts:")
print("actual_mw:", int(baseline_actual_test.isna().sum()))
print("forecast_mw:", int(baseline_forecast_test.isna().sum()))

Split date: 2025-01-01 00:00:00+00:00
Train time rows: 36139
Test time rows: 6316

X_train shape: (35222, 50)
X_test shape: (6233, 50)
y_train shape: (35222,)
y_test shape: (6233,)

Top NaN counts in X_train:
forecast_capacity_ratio_safe    25726
forecast_capacity_ratio         25726
capacity_ffill_safe             23245
capacity_mw                     23245
forecast_roll_mean_24              12
forecast_roll_mean_3               12
forecast_roll_mean_6               12
forecast_diff_1                    12
forecast_ramp_abs                  12
shortwave_roll_mean_6               1
shortwave_roll_mean_3               1
shortwave_roll_mean_24              1
shortwave_diff_1                    1
shortwave_ramp_abs                  1
cloud_cover_frac                    0
capacity_ffill                      0
capacity_missing_flag               0
shortwave_x_temp                    0
forecast_mw_log1p                   0
shortwave_radiation_log1p           0
forecast_x_hour_sin            

In [30]:
assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert baseline_actual_test.shape[0] == baseline_forecast_test.shape[0] == y_test.shape[0]
assert y_train.notna().all()
assert y_test.notna().all()
assert baseline_actual_test.notna().all()
assert baseline_forecast_test.notna().all()

## Baselines

In [31]:
numeric_features = X_train.columns.tolist()

ridge_preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        )
    ],
    remainder="drop",
)

ridge_model = Pipeline(
    steps=[
        ("prep", ridge_preprocess),
        ("model", Ridge(alpha=1.0)),
    ]
)

ridge_model.fit(X_train, y_train)

ridge_residual_pred = ridge_model.predict(X_test)
ridge_corrected = baseline_forecast_test + ridge_residual_pred

ridge_mae = mean_absolute_error(baseline_actual_test, ridge_corrected)
ridge_rmse = np.sqrt(mean_squared_error(baseline_actual_test, ridge_corrected))

print("Ridge Residual Model")
print("MAE:", ridge_mae)
print("RMSE:", ridge_rmse)

Ridge Residual Model
MAE: 116.91484336243482
RMSE: 210.16820165275124


In [32]:
tree_preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            numeric_features,
        )
    ],
    remainder="drop",
)

tree_model = Pipeline(
    steps=[
        ("prep", tree_preprocess),
        (
            "model",
            HistGradientBoostingRegressor(
                max_depth=6,
                learning_rate=0.05,
                max_iter=300,
                random_state=42,
            ),
        ),
    ]
)

tree_model.fit(X_train, y_train)

tree_residual_pred = tree_model.predict(X_test)
tree_corrected = baseline_forecast_test + tree_residual_pred

tree_mae = mean_absolute_error(baseline_actual_test, tree_corrected)
tree_rmse = np.sqrt(mean_squared_error(baseline_actual_test, tree_corrected))

print("Gradient Boosted Residual Model")
print("MAE:", tree_mae)
print("RMSE:", tree_rmse)

Gradient Boosted Residual Model
MAE: 117.79342621239691
RMSE: 217.17404881239617


## Results and Interpretations

In [33]:
results = pd.DataFrame(
    {
        "Model": [
            "NYISO Baseline",
            "Ridge Residual",
            "Gradient Boosted Residual",
        ],
        "MAE": [
            baseline_mae,
            ridge_mae,
            tree_mae,
        ],
        "RMSE": [
            baseline_rmse,
            ridge_rmse,
            tree_rmse,
        ],
    }
)

results

,Model,MAE,RMSE
0,NYISO Baseline,114.324099,224.168630
1,Ridge Residual,116.914843,210.168202
2,Gradient Boosted Residual,117.793426,217.174049
